# 04 - 推理与评估

本 notebook 讲解：
1. `eval_omni.py` 的推理流程
2. `scripts/batch_validate_omni.py` 批量验证
3. `scripts/compare_eval_runs.py` 对比两个评估结果
4. `scripts/asr_eval_generated_audio.py` ASR 回译评估
5. 评估指标的含义与局限

## 1. eval_omni.py 推理流程

`eval_omni.py` 是交互式/演示推理入口，支持：
- 加载原生 torch 权重或 transformers 格式模型
- text / audio / image / clone / mix 多种模式
- 流式输出文本 + 音频
- 把 Mimi codes 解码为 mp3

核心函数：
- `init_model(args)`：加载 tokenizer、模型、Mimi decoder
- `eval_sample(...)`：构造 prompt，调用 `model.generate(stream=True)`，解码音频

In [ ]:
import os, sys
ROOT = "/home/genesis/Projects/minimind-o"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# 检查可用权重
import glob
weights = sorted(glob.glob("out/*_768.pth"))
print("可用 768 hidden 权重:")
for w in weights:
    print(f"  {w}")

### 1.1 推理 dtype 选择

代码见 `eval_omni.py:40-42`。

- 默认 `bf16`
- `fp16` 在修复前会导致非有限概率（`model_omni.py` 的 `generate` 已添加 `.float()`）
- `fp32` 最稳但最慢

**新人注意**：如果看到 `RuntimeError: probability tensor contains either inf, nan or element < 0`，先检查 dtype 是否为 bf16 或 fp32。

### 1.2 audio decode 过滤

代码见 `eval_omni.py:80-82`：
```python
filtered = torch.where(mimi_codes >= 2049, torch.zeros_like(mimi_codes), mimi_codes)
audio = model.mimi_model.decode(filtered).audio_values
```

Mimi 有效 code 范围是 0~2047；2048 以上是特殊 token。解码前必须过滤掉，否则 Mimi 可能报错或输出噪声。

## 2. batch_validate_omni.py

代码见 `scripts/batch_validate_omni.py`。

输入：一个 `jsonl` 测试集，每个 case 包含：
- `id`, `type`（text/audio/image）
- `prompt`（text 用）
- `audio_path`（audio 用）
- `image_path`（image 用）
- `expect_audio`, `min_chars`, `min_audio_frames` 等检查字段

输出：
- `results.jsonl`：每条详细结果
- `summary.json` / `summary.md`：聚合报告

In [ ]:
# 检查测试集
import glob
test_sets = sorted(glob.glob("dataset/*.jsonl") + glob.glob("dataset/**/*.jsonl", recursive=True))
print("可用测试集:")
for t in test_sets[:20]:
    print(f"  {t}")

### 2.1 Basic pass 的定义

`batch_validate_omni.py` 的 pass 只检查：
- 文本非空且长度 >= min_chars
- 4-gram repeat score <= max_repeat_score（默认 0.35）
- 音频 frame 数量足够（如果 expect_audio=True）
- 权重加载无 missing/unexpected keys

**不代表语义正确、音质优秀、图像理解准确**。

### 2.2 运行一次批量验证

下面给出命令模板。需要确保 `out/{weight}_768.pth` 存在。

In [ ]:
weight = "sft_full_muon_final" if os.path.exists("out/sft_full_muon_final_768.pth") else "sft_zero_muon"
test_set = "dataset/eval_muon_mini.jsonl" if os.path.exists("dataset/eval_muon_mini.jsonl") else "dataset/eval_omni/eval_cases.jsonl"

cmd = (
    f"CUDA_VISIBLE_DEVICES=0 python scripts/batch_validate_omni.py "
    f"--test_set {test_set} --output_dir eval_results/nb04_smoke "
    f"--weight {weight} --max_new_tokens 96 --dtype bf16 --decode_audio"
)
print("批量验证命令:")
print(cmd)
print("\n请在终端运行上述命令，然后读取 eval_results/nb04_smoke/summary.md 查看结果。")

## 3. compare_eval_runs.py

代码见 `scripts/compare_eval_runs.py`。

输入两个 `batch_validate_omni.py` 的输出目录，生成 markdown 对比报告。
对比维度：
- overall pass rate
- by type 的平均 chars / frames / repeat
- per-case 的回归/改进标记

In [ ]:
# 对比命令模板
cmd = (
    "python scripts/compare_eval_runs.py "
    "--run_a eval_results/sft_full_muon_final_l1_bf16_t160 "
    "--run_b eval_results/sft_full_muon_v3_final_l1_bf16_t160 "
    "--label_a baseline_A "
    "--label_b runC_v3 "
    "--output docs/evaluation_results/compare_nb04_example.md"
)
print(cmd)

## 4. ASR 回译评估

`scripts/asr_eval_generated_audio.py` 把 batch_validate 生成的音频再用 SenseVoice 转录，计算 CER/WER。

这是判断 audio 可懂度的核心代理指标。

**注意**：
- CER 低不代表语义正确。典型案例：`a2a_en_black_hole` CER=0 但语义完全错误。
- 中文短句 WER 噪声大，优先看 CER。
- 需要与人工听感和语义 review 结合使用。

In [ ]:
# ASR 评估命令模板
cmd = (
    "python scripts/asr_eval_generated_audio.py "
    "--results eval_results/sft_full_muon_v3_final_l1_bf16_t160/results.jsonl "
    "--output eval_results/sft_full_muon_v3_final_l1_bf16_t160/asr_eval.json "
    "--markdown eval_results/sft_full_muon_v3_final_l1_bf16_t160/asr_eval.md "
    "--backend sensevoice "
    "--sensevoice_project /home/genesis/Projects/SenseVoice "
    "--sensevoice_model model/SenseVoiceSmall "
    "--device cpu --batch_size 16"
)
print(cmd)

## 5. 评估指标的解读

| 指标 | 含义 | 优点 | 局限 |
| --- | --- | --- | --- |
| Basic pass | 结构检查（非空、不重复、音频帧够） | 快速、自动化 | 不衡量质量 |
| CER | 字符错误率 | 细粒度、对短中文友好 | 可能被语义错误但发音相近的样本拉低 |
| WER | 词错误率 | 对英文自然 | 中文短句噪声大 |
| Repeat score | 4-gram 重复度 | 检测复读机 | 阈值敏感 |
| Special code rate | 非法/特殊 audio code 占比 | 检测解码污染 | 低不代表音质好 |
| AI review | LLM/人工判断语义 | 更接近真实质量 | 主观、成本高 |

当前仓库的评估升级方向：
- L1 从 57 case 扩展到 100+
- 增加语义评分（与 CER 分离）
- 增加人工听感 spot-check